In [3]:
import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm  # для отображения прогресса, установите: pip install tqdm

from groundingdino.util.inference import load_model, load_image, predict
import sys
sys.path.append(os.path.abspath('..'))
from noises.wrapper import add_noise 

### Загрузка модели

In [8]:
config_path = "/Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
weights_path = "/Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/GroundingDINO/weights/groundingdino_swint_ogc.pth"

model = load_model(config_path, weights_path)
model = model.to('cpu')   # или 'cuda' если есть GPU

text_prompt = "people"
box_threshold = 0.35
text_threshold = 0.25

final text_encoder_type: bert-base-uncased


### Инференс на 1 изображении

In [6]:
def count_people_in_image(image_path, model):
    """Возвращает количество обнаруженных людей на изображении."""
    try:
        image_source, image = load_image(image_path)
        boxes, logits, phrases = predict(
            model=model,
            image=image,
            caption=text_prompt,
            box_threshold=box_threshold,
            text_threshold=text_threshold,
            device="cpu"
        )
        return len(boxes)
    except Exception as e:
        print(f"Ошибка при обработке {image_path}: {e}")
        return 0

### Обработка всех кадров

In [4]:
def process_frames(frame_dir, output_npy_path, noise_params=None, skip_existing=True):
    """
    Обрабатывает все кадры из frame_dir.
    Если noise_params задан, применяет шум через add_noise перед инференсом.

    Параметры:
        frame_dir : str - папка с кадрами (имена seq_XXXXXX.jpg)
        output_npy_path : str - путь для сохранения результата
        noise_params : dict - параметры для add_noise, например:
            {"noise_type": "gaussian", "mean": 0, "var": 0.1}
            или None (без шума)
        skip_existing : bool - если True и файл существует, загружает его
    """
    if skip_existing and os.path.exists(output_npy_path):
        print(f"Загружено из существующего: {output_npy_path}")
        return np.load(output_npy_path)

    frame_files = sorted([f for f in os.listdir(frame_dir) if f.endswith('.jpg')])
    counts = []

    temp_dir = None
    if noise_params is not None:
        temp_dir = Path("temp_noisy_frames")
        temp_dir.mkdir(exist_ok=True)

    for frame_file in tqdm(frame_files, desc=f"Обработка {Path(output_npy_path).name}"):
        img_path = os.path.join(frame_dir, frame_file)

        if noise_params is not None:
            img = cv2.imread(img_path)
            if img is None:
                print(f"Не удалось прочитать {img_path}")
                counts.append(0)
                continue
            noisy_img = add_noise(img, **noise_params)  # применение шума
            temp_img_path = temp_dir / frame_file
            cv2.imwrite(str(temp_img_path), noisy_img)
            cnt = count_people_in_image(str(temp_img_path), model)
        else:
            cnt = count_people_in_image(img_path, model)

        counts.append(cnt)

    if temp_dir is not None:
        for f in temp_dir.iterdir():
            f.unlink()
        temp_dir.rmdir()

    counts = np.array(counts, dtype=int)
    np.save(output_npy_path, counts)
    print(f"Сохранено: {output_npy_path}")
    return counts

### Запуск

In [ ]:
frame_dir = "/Users/v.makshanchikov/Documents/Python Proj/ВКР Магистратура/mall_dataset/pre_frames"
output_dir = Path("groundingdino_counts")
output_dir.mkdir(exist_ok=True)

# # 4.1 Без шума
# counts_original = process_frames(
#     frame_dir,
#     output_npy_path=str(output_dir / "counts_original.npy"),
#     noise_params=None,
#     skip_existing=True
# )

# 4.2 Гауссовский шум
counts_gaussian = process_frames(
    frame_dir,
    output_npy_path=str(output_dir / "counts_gaussian.npy"),
    noise_params={"noise_type": "gaussian", "mean": 0.0, "var": 0.01},
    skip_existing=True
)

# 4.3 Соль и перец
counts_sp = process_frames(
    frame_dir,
    output_npy_path=str(output_dir / "counts_salt_pepper.npy"),
    noise_params={"noise_type": "salt_pepper", "salt_prob": 0.01, "pepper_prob": 0.01},
    skip_existing=True
)

# 4.4 Спекл-шум
counts_speckle = process_frames(
    frame_dir,
    output_npy_path=str(output_dir / "counts_speckle.npy"),
    noise_params={"noise_type": "speckle", "speckle_scale": 0.2},
    skip_existing=True
)

# 4.5 Пуассоновский шум
counts_poisson = process_frames(
    frame_dir,
    output_npy_path=str(output_dir / "counts_poisson.npy"),
    noise_params={"noise_type": "poisson", "poisson_scale": 1.0},
    skip_existing=True
)

print("Все массивы успешно сохранены.")

Обработка counts_gaussian.npy: 100%|██████████| 5/5 [00:18<00:00,  3.77s/it]

Сохранено: groundingdino_counts/counts_gaussian.npy


In [14]:
states_gaussian = np.load("groundingdino_counts/counts_gaussian.npy")
states_gaussian

array([18, 20, 21, 22, 19])